# Section 07 guided demonstrations — PySpark optimizations

This notebook is demonstrated by the instructor. It is not an independent lab or project deliverable.

Run the setup once, then observe one idea at a time. The datasets are deliberately small so that the plan and behaviour remain easy to explain.


## Demonstration setup — run once

Create one Spark session and make the adaptive settings visible.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Optimization-Demonstrations')
         .config('spark.sql.shuffle.partitions', '20')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
         .config('spark.sql.adaptive.skewJoin.enabled', 'true')
         .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark:', spark.version, '| master:', spark.sparkContext.master)
print('AQE:', spark.conf.get('spark.sql.adaptive.enabled'))


## Demo 1 — See partitions as groups of rows

A task normally processes one partition in one stage. Display the partition identifier to make the invisible distribution visible.


In [ ]:
partitioned = spark.range(0, 24, 1, 4).withColumn('bucket', F.col('id') % 3)
print('Starting partitions:', partitioned.rdd.getNumPartitions())

partition_layout = (partitioned.withColumn('partition_id', F.spark_partition_id())
                    .groupBy('partition_id')
                    .count()
                    .orderBy('partition_id'))
partition_layout.show()


### What to notice

- Four input partitions give Spark four pieces of work.
- The row count does not change when the partition layout changes.
- Partition count is about distribution, not business grouping.


## Demo 2 — Compare repartition and coalesce

`repartition` redistributes rows and can increase or decrease the count. `coalesce` is mainly used to reduce the count with less movement.


In [ ]:
repartitioned = partitioned.repartition(6, 'bucket')
coalesced = repartitioned.coalesce(2)

print('Original:', partitioned.rdd.getNumPartitions())
print('After repartition:', repartitioned.rdd.getNumPartitions())
print('After coalesce:', coalesced.rdd.getNumPartitions())

(repartitioned.withColumn('partition_id', F.spark_partition_id())
 .groupBy('partition_id', 'bucket').count()
 .orderBy('partition_id', 'bucket').show())


### What to notice

- Repartitioning by `bucket` places matching bucket values together.
- Repartitioning introduces an exchange because rows move between partitions.
- Coalescing is suitable after data becomes smaller; forcing one partition can remove useful parallelism.


In [ ]:
repartitioned.explain('formatted')


## Demo 3 — Cache and persist lifecycle

Persist an expensive intermediate only when it will be reused. The first action materializes it; `unpersist` releases it.


In [ ]:
reused = (spark.range(0, 100_000, 1, 8)
          .withColumn('group_id', F.col('id') % 20)
          .withColumn('calculated_value', F.sqrt(F.col('id') + 1))
          .persist(StorageLevel.MEMORY_AND_DISK))

print('Storage before action:', reused.storageLevel)
print('Rows materialized:', reused.count())
reused.groupBy('group_id').agg(F.avg('calculated_value').alias('average')).show(5)
reused.filter(F.col('id') < 5).show()
reused.explain('formatted')


### Release the persisted result

Leaving old caches in place makes them compete with useful data for executor storage.


In [ ]:
reused.unpersist()
print('Storage after unpersist:', reused.storageLevel)


## Demo 4 — Broadcast a small join side

A small product table can be copied to each executor so the larger transaction side does not need a full join shuffle.


In [ ]:
transactions = spark.createDataFrame([
    ('T001', 'P01', 2),
    ('T002', 'P02', 1),
    ('T003', 'P01', 4),
    ('T004', 'P03', 3),
    ('T005', 'P99', 1),
], ['transaction_id', 'product_id', 'quantity'])

products = spark.createDataFrame([
    ('P01', 'Keyboard', 'Accessories'),
    ('P02', 'Headphones', 'Audio'),
    ('P03', 'Monitor', 'Electronics'),
], ['product_id', 'product_name', 'category'])

broadcast_result = transactions.join(F.broadcast(products), 'product_id', 'left')
broadcast_result.show(truncate=False)
broadcast_result.explain('formatted')


### What to notice

Look for `BroadcastHashJoin` and `BroadcastExchange`. The hint is appropriate only when the broadcast side is safely small.


## Demo 5 — Adaptive Query Execution

AQE uses runtime statistics to improve parts of the physical plan after execution begins. Run an action before inspecting the final adaptive plan.


In [ ]:
for setting in [
    'spark.sql.adaptive.enabled',
    'spark.sql.adaptive.coalescePartitions.enabled',
    'spark.sql.adaptive.skewJoin.enabled',
]:
    print(setting, '=', spark.conf.get(setting))

adaptive_summary = (spark.range(0, 100_000, 1, 40)
                    .withColumn('group_id', F.col('id') % 10)
                    .groupBy('group_id')
                    .agg(F.count('*').alias('rows')))

adaptive_summary.collect()
adaptive_summary.explain('formatted')


### What to notice

- Look for `AdaptiveSparkPlan`.
- Compare the initial and final plan where the output provides both.
- AQE can combine small shuffle partitions and handle certain skewed joins, but it does not repair inefficient business logic.


## Demo 6 — Make skew visible

One popular key can place much more work in one partition than the others.


In [ ]:
skewed_rows = [('POPULAR', value) for value in range(100)] + [(f'K{value}', value) for value in range(10)]
skewed = spark.createDataFrame(skewed_rows, ['customer_id', 'value'])
(skewed.groupBy('customer_id').count()
 .orderBy(F.desc('count'), F.asc('customer_id'))
 .show(12, truncate=False))


### What to notice

The `POPULAR` key is much larger than the others. Confirm skew with data and task metrics before considering broadcast, pre-aggregation, AQE skew handling or advanced techniques such as salting.


## Demo 7 — Reduce work early and use built-in expressions

Keep only needed rows and columns before expensive operations. Built-in expressions remain visible to Spark's optimizer.


In [ ]:
source = spark.createDataFrame([
    ('T01', ' completed ', 250.0, 'unused-a'),
    ('T02', 'cancelled', 100.0, 'unused-b'),
    ('T03', 'COMPLETED', -5.0, 'unused-c'),
], ['transaction_id', 'status', 'amount', 'unused_payload'])

reduced = (source
           .select('transaction_id', 'status', 'amount')
           .withColumn('normalized_status', F.upper(F.trim('status')))
           .filter((F.col('normalized_status') == 'COMPLETED') & (F.col('amount') > 0)))

reduced.show(truncate=False)
reduced.explain('formatted')


## Demonstration close

Before changing a job, identify the expensive stage, inspect its plan and partitions, make one meaningful change and measure again.

Continue to [Section 08 — Guided e-commerce pipeline](../08-guided-project/README.md).
